# Generative Models to Estimate the Gradients of the Data Distribution :: MNIST Dataset

Source: <BR>
https://github.com/JeongJiHeon/ScoreDiffusionModel/tree/main/NCSN

Adapted:

Antonio Esteves @UMinho, February 2025

---

## Model Definition

First, we defined the perturbed data distribution.

$$
q_{\sigma}(x)=\int p \mathcal(x \vert t, \sigma^2\mathbf{I}) d\sigma dt \tag{1}
$$

However, $s_{\theta}(x) = \nabla_{x} \log q_{\sigma}(x) \approx \nabla_{x} \log p(x)$ is true only when the noise is small enough, such that, $q_{\sigma}(x) \approx p(x)$.
By adding noise with multiple variances, we obtain a sequence of noise-perturbed distributions that converge to the true data distribution.
<br>

The conditional perturbed data distribution for step $i$ is: <BR>

$$
q_{\sigma_{i}}(\tilde{x} \vert x)=\mathcal{N}(\tilde{x} ; x, \sigma_{i}^2\mathbf{I}) \tag{2}
$$

This distribution can be reparameterized as: <BR>

$$
\tilde{x}=x+\sigma_{i}*z \tag{3}
$$

where<BR>

$$
z \sim \mathcal{N}(0, \mathbf{I}) \tag{4}
$$

and in each step the noise variance is bigger than in the next step:<BR>

$$
\frac{\sigma_{1}}{\sigma_{2}}=\frac{\sigma_{2}}{\sigma_{3}}=\cdots=\frac{\sigma_{L-1}}{\sigma_{L}} > 1 \tag{5}
$$

Next, we define a score model that depends on the variances.

$$
\mathbf{s}_{\theta}(\tilde{x}, \sigma_i) \approx \nabla_{\tilde{x}} \log{q_{\sigma_i} }(\tilde{x}\vert{x}) \tag{6}
$$


Using the expression of a Gaussian with mean $x$ and variance $\sigma_i^2$, the score of perturbed data distribution is:

$$
\begin{align}
\begin{aligned}
\nabla_{\tilde{x}}\log q_{\sigma_i}(\tilde{x}\vert{x}) & = 
\frac{d}{d\tilde{x}} \log \left( constant*e^{-\frac{(\tilde{x}-x)^2}{2\sigma_i^2}} \right) \\
& = - \frac{1}{2}\frac{d}{d\tilde{x}} \left(\frac{(\tilde{x}-x)^2}{\sigma_i^2} \right) \\
& = - \frac{2}{2} \frac{\tilde{x}-x}{\sigma_i} \frac{d}{d\tilde{x}} \frac{\tilde{x}-x}{\sigma_i} \\
& = - \frac{(\tilde{x}-x)}{\sigma_i^2} \\
& = -\frac{z}{\sigma_i} \text{(using equation 3, } z = \frac{\tilde{x}-x}{\sigma_i} \text{)}
\end{aligned}
\tag{7}
\end{align}
$$ 

Then, we can use the objective function for noise conditional score network via score matching.

$$
\mathcal{L}(\theta, \{ \sigma_{i}\}_{i=1}^L) = \frac{1}{L}\sum_{i=1}^L\lambda({\sigma_i})\mathcal{l}(\theta;\sigma_i) \tag{8}
$$

where

$$\mathcal{l}(\theta;\sigma_i) = \frac{1}{2}\mathbb{E}_{x \sim p(x)} \mathbb{E}_{\tilde{x}\sim\mathcal{N}(x,\sigma_i^2\mathbf{I})} \left[ \Vert \mathbf{s}_{\theta}(\tilde{x}, \sigma_i)+\frac{\tilde{x}-x}{\sigma_i^2}\Vert^2_2 \right] \tag{9}
$$

and we will opt for $\lambda$ given by

$$\lambda(\sigma_i) = \sigma_i^2 \tag{10}$$


## Auxiliary Layers of the Score Model 

In [ ]:
import math
import torch
import torch
import torchvision
import torch.nn                    as nn
import matplotlib.pyplot           as     plt
import matplotlib.animation        as     animation
from   scipy.ndimage.interpolation import rotate
import numpy                       as     np
from   IPython.display             import HTML
from   IPython.display             import clear_output
import torch.nn                    as     nn
import torch.nn.functional         as     F
from   functools                   import partial


In [ ]:
def conv3x3(in_planes, out_planes, stride=1, bias=False):
    '''
    Convolution layer with 3x3 filters and padding.
    '''
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=bias,
    )

def conv1x1(in_planes, out_planes, stride=1, bias=False):
    '''
    Convolution layer with 1x1 filters.
    '''
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=1,
        stride=stride,
        padding=0,
        bias=bias,
    )

def dilated_conv3x3(in_planes, out_planes, dilation, bias=True):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, padding=dilation, dilation=dilation, bias=bias)


class ConditionalBatchNorm2d(nn.Module):
    '''
    Batch normalization layer where the normalization 'gamma' factor is given 
    by the input 'y'.
    Data is normalized using statistics computed per channel over all batch samples.
    
    out = (x-E[x])/(sqrt(Var[x]+epsilon) * gamma + beta
    '''
    def __init__(self, num_features, num_classes, bias=True):
        super().__init__()
        self.num_features = num_features
        self.bias         = bias
        self.bn           = nn.BatchNorm2d(num_features, affine=False)
        if self.bias:
            self.embed    = nn.Embedding(num_classes, num_features * 2)
            self.embed.weight.data[:, :num_features].uniform_()  # Initialise scale at N(1, 0.02)
            self.embed.weight.data[:, num_features:].zero_()     # Initialise bias at 0
        else:
            self.embed = nn.Embedding(num_classes, num_features)
            self.embed.weight.data.uniform_()

    def forward(self, x, y):
        out = self.bn(x)
        if self.bias:
            gamma, beta = self.embed(y).chunk(2, dim=1)
            out         = gamma.view(-1, self.num_features, 1, 1) * out + beta.view(-1, self.num_features, 1, 1)
        else:
            gamma = self.embed(y)
            out   = gamma.view(-1, self.num_features, 1, 1) * out
        return out


class ConditionalInstanceNorm2d(nn.Module):
    '''
    Instance normalization layer where the normalization 'gamma' factor is given 
    by the input 'y'.
    The mean and standard deviation are calculated per dimension (C,H,W) separately 
    for each sample in a mini-batch
    
    out = (x-E[x])/(sqrt(Var[x]+epsilon) * gamma + beta
    '''
    def __init__(self, num_features, num_classes, bias=True):
        super().__init__()
        self.num_features  = num_features
        self.bias          = bias
        self.instance_norm = nn.InstanceNorm2d(num_features, affine=False, track_running_stats=False)
        if bias:
            self.embed = nn.Embedding(num_classes, num_features * 2)
            self.embed.weight.data[:, :num_features].uniform_()  # Initialise scale at N(1, 0.02)
            self.embed.weight.data[:, num_features:].zero_()  # Initialise bias at 0
        else:
            self.embed = nn.Embedding(num_classes, num_features)
            self.embed.weight.data.uniform_()

    def forward(self, x, y):
        h = self.instance_norm(x)
        if self.bias:
            gamma, beta = self.embed(y).chunk(2, dim=-1)
            out = gamma.view(-1, self.num_features, 1, 1) * h + beta.view(-1, self.num_features, 1, 1)
        else:
            gamma = self.embed(y)
            out = gamma.view(-1, self.num_features, 1, 1) * h
        return out


class CRPBlock(nn.Module):
    def __init__(self, features, n_stages, act=nn.ReLU()):
        super().__init__()
        self.convs = nn.ModuleList()
        for i in range(n_stages):
            self.convs.append(conv3x3(features, features, stride=1, bias=False))
        self.n_stages = n_stages
        self.maxpool = nn.MaxPool2d(kernel_size=5, stride=1, padding=2)
        self.act = act

    def forward(self, x):
        x = self.act(x)
        path = x
        for i in range(self.n_stages):
            path = self.maxpool(path)
            path = self.convs[i](path)
            x = path + x
        return x


class CondCRPBlock(nn.Module):
    def __init__(self, features, n_stages, num_classes, normalizer, act=nn.ReLU()):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(n_stages):
            self.norms.append(normalizer(features, num_classes, bias=True))
            self.convs.append(conv3x3(features, features, stride=1, bias=False))
        self.n_stages = n_stages
        self.maxpool = nn.AvgPool2d(kernel_size=5, stride=1, padding=2)
        self.act = act

    def forward(self, x, y):
        x = self.act(x)
        path = x
        for i in range(self.n_stages):
            path = self.norms[i](path, y)
            path = self.maxpool(path)
            path = self.convs[i](path)
            x    = path + x
        return x


class CondRCUBlock(nn.Module):
    def __init__(self, features, n_blocks, n_stages, num_classes, normalizer, act=nn.ReLU()):
        super().__init__()

        for i in range(n_blocks):
            for j in range(n_stages):
                setattr(self, '{}_{}_norm'.format(i + 1, j + 1), normalizer(features, num_classes, bias=True))
                setattr(self, '{}_{}_conv'.format(i + 1, j + 1),
                        conv3x3(features, features, stride=1, bias=False))

        self.stride   = 1
        self.n_blocks = n_blocks
        self.n_stages = n_stages
        self.act      = act

    def forward(self, x, y):
        for i in range(self.n_blocks):
            residual = x
            for j in range(self.n_stages):
                x = getattr(self, '{}_{}_norm'.format(i + 1, j + 1))(x, y)
                x = self.act(x)
                x = getattr(self, '{}_{}_conv'.format(i + 1, j + 1))(x)
            x += residual
        return x


class CondMSFBlock(nn.Module):
    def __init__(self, in_planes, features, num_classes, normalizer):
        """
        :param in_planes: tuples of input planes
        """
        super().__init__()
        assert isinstance(in_planes, list) or isinstance(in_planes, tuple)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.features = features

        for i in range(len(in_planes)):
            self.convs.append(conv3x3(in_planes[i], features, stride=1, bias=True))
            self.norms.append(normalizer(in_planes[i], num_classes, bias=True))

    def forward(self, xs, y, shape):
        sums = torch.zeros(xs[0].shape[0], self.features, *shape, device=xs[0].device)
        for i in range(len(self.convs)):
            h = self.norms[i](xs[i], y)
            h = self.convs[i](h)
            h = F.interpolate(h, size=shape, mode='bilinear', align_corners=True)
            sums += h
        return sums


class CondRefineBlock(nn.Module):
    def __init__(self, in_planes, features, num_classes, normalizer, act=nn.ReLU(), start=False, end=False):
        super().__init__()

        assert isinstance(in_planes, tuple) or isinstance(in_planes, list)
        self.n_blocks = n_blocks = len(in_planes)

        self.adapt_convs = nn.ModuleList()
        for i in range(n_blocks):
            self.adapt_convs.append(
                CondRCUBlock(in_planes[i], 2, 2, num_classes, normalizer, act)
            )

        self.output_convs = CondRCUBlock(features, 3 if end else 1, 2, num_classes, normalizer, act)

        if not start:
            self.msf = CondMSFBlock(in_planes, features, num_classes, normalizer)

        self.crp = CondCRPBlock(features, 2, num_classes, normalizer, act)

    def forward(self, xs, y, output_shape):
        assert isinstance(xs, tuple) or isinstance(xs, list)
        hs = []
        for i in range(len(xs)):
            h = self.adapt_convs[i](xs[i], y)
            hs.append(h)

        if self.n_blocks > 1:
            h = self.msf(hs, y, output_shape)
        else:
            h = hs[0]

        h = self.crp(h, y)
        h = self.output_convs(h, y)

        return h


class ConvMeanPool(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, biases=True, adjust_padding=False):
        super().__init__()
        if not adjust_padding:
            self.conv = nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)
        else:
            self.conv = nn.Sequential(
                nn.ZeroPad2d((1, 0, 1, 0)),
                nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)
            )

    def forward(self, inputs):
        output = self.conv(inputs)
        output = sum(
            [output[:, :, ::2, ::2], output[:, :, 1::2, ::2], output[:, :, ::2, 1::2], output[:, :, 1::2, 1::2]]) / 4.
        return output


class MeanPoolConv(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, biases=True):
        super().__init__()
        self.conv = nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)

    def forward(self, inputs):
        output = inputs
        output = sum(
            [output[:, :, ::2, ::2], output[:, :, 1::2, ::2], output[:, :, ::2, 1::2], output[:, :, 1::2, 1::2]]) / 4.
        return self.conv(output)


class UpsampleConv(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, biases=True):
        super().__init__()
        self.conv = nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)
        self.pixelshuffle = nn.PixelShuffle(upscale_factor=2)

    def forward(self, inputs):
        output = inputs
        output = torch.cat([output, output, output, output], dim=1)
        output = self.pixelshuffle(output)
        return self.conv(output)


class ConditionalResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim, num_classes, resample=None, act=nn.ELU(),
                 normalization=ConditionalBatchNorm2d, adjust_padding=False, dilation=None):
        super().__init__()
        self.non_linearity = act
        self.input_dim     = input_dim
        self.output_dim    = output_dim
        self.resample      = resample
        if resample == 'down':
            if dilation is not None:
                self.conv1      = dilated_conv3x3(input_dim, input_dim, dilation=dilation)
                self.normalize2 = normalization(input_dim, num_classes)
                self.conv2      = dilated_conv3x3(input_dim, output_dim, dilation=dilation)
                conv_shortcut   = partial(dilated_conv3x3, dilation=dilation)
            else:
                self.conv1      = nn.Conv2d(input_dim, input_dim, 3, stride=1, padding=1)
                self.normalize2 = normalization(input_dim, num_classes)
                self.conv2      = ConvMeanPool(input_dim, output_dim, 3, adjust_padding=adjust_padding)
                conv_shortcut   = partial(ConvMeanPool, kernel_size=1, adjust_padding=adjust_padding)

        elif resample is None:
            if dilation is not None:
                conv_shortcut   = partial(dilated_conv3x3, dilation=dilation)
                self.conv1      = dilated_conv3x3(input_dim, output_dim, dilation=dilation)
                self.normalize2 = normalization(output_dim, num_classes)
                self.conv2      = dilated_conv3x3(output_dim, output_dim, dilation=dilation)
            else:
                conv_shortcut   = nn.Conv2d
                self.conv1      = nn.Conv2d(input_dim, output_dim, kernel_size=3, stride=1, padding=1)
                self.normalize2 = normalization(output_dim, num_classes)
                self.conv2      = nn.Conv2d(output_dim, output_dim, kernel_size=3, stride=1, padding=1)
        else:
            raise Exception('invalid resample value')

        if output_dim != input_dim or resample is not None:
            self.shortcut = conv_shortcut(input_dim, output_dim)

        self.normalize1 = normalization(input_dim, num_classes)

    def forward(self, x, y):
        output = self.normalize1(x, y)
        output = self.non_linearity(output)
        output = self.conv1(output)
        output = self.normalize2(output, y)
        output = self.non_linearity(output)
        output = self.conv2(output)

        if self.output_dim == self.input_dim and self.resample is None:
            shortcut = x
        else:
            shortcut = self.shortcut(x)

        return shortcut + output


class ConditionalInstanceNorm2dPlus(nn.Module):
    def __init__(self, num_features, num_classes, bias=True):
        super().__init__()
        self.num_features  = num_features
        self.bias          = bias
        self.instance_norm = nn.InstanceNorm2d(num_features, affine=False, track_running_stats=False)
        if bias:
            self.embed = nn.Embedding(num_classes, num_features * 3)
            self.embed.weight.data[:, :2 * num_features].normal_(1, 0.02)  # Initialise scale at N(1, 0.02)
            self.embed.weight.data[:, 2 * num_features:].zero_()           # Initialise bias at 0
        else:
            self.embed = nn.Embedding(num_classes, 2 * num_features)
            self.embed.weight.data.normal_(1, 0.02)

    def forward(self, x, y):
        means = torch.mean(x, dim=(2, 3))
        m     = torch.mean(means, dim=-1, keepdim=True)
        v     = torch.var(means, dim=-1, keepdim=True)
        means = (means - m) / (torch.sqrt(v + 1e-5))
        h     = self.instance_norm(x)

        if self.bias:
            gamma, alpha, beta = self.embed(y).chunk(3, dim=-1)
            h   = h + means[..., None, None] * alpha[..., None, None]
            out = gamma.view(-1, self.num_features, 1, 1) * h + beta.view(-1, self.num_features, 1, 1)
        else:
            gamma, alpha = self.embed(y).chunk(2, dim=-1)
            h   = h + means[..., None, None] * alpha[..., None, None]
            out = gamma.view(-1, self.num_features, 1, 1) * h
        return out


class CondRefineNetDilated(nn.Module):
    def __init__(self,  device, L):
        super().__init__()
        # self.norm      = ConditionalInstanceNorm2d
        self.norm        = ConditionalInstanceNorm2dPlus
        self.ngf         = 64
        self.num_classes = L
        self.act         = act = nn.ELU()
        self.device      = device
        # self.act       = act = nn.ReLU(True)

        self.begin_conv = nn.Conv2d(1, self.ngf, 3, stride=1, padding=1)
        self.normalizer = self.norm(self.ngf, self.num_classes)
        self.end_conv   = nn.Conv2d(self.ngf, 1, 3, stride=1, padding=1)

        self.res1 = nn.ModuleList(
            [
            ConditionalResidualBlock(
                self.ngf,
                self.ngf,
                self.num_classes,
                resample      = None,
                act           = act,
                normalization = self.norm,
                ),
            ConditionalResidualBlock(
                self.ngf,
                self.ngf,
                self.num_classes,
                resample      = None,
                act           = act,
                normalization = self.norm,
                )
            ]
        )

        self.res2 = nn.ModuleList(
            [
            ConditionalResidualBlock(
                self.ngf,
                2 * self.ngf,
                self.num_classes,
                resample      = 'down',
                act           = act,
                normalization = self.norm,
                ),
            ConditionalResidualBlock(
                2 * self.ngf,
                2 * self.ngf,
                self.num_classes,
                resample      = None,
                act           = act,
                normalization = self.norm,
                )
            ]
        )

        self.res3 = nn.ModuleList(
            [
            ConditionalResidualBlock(
                2 * self.ngf,
                2 * self.ngf,
                self.num_classes,
                resample      = 'down',
                act           = act,
                normalization = self.norm,
                dilation      = 2,
                ),
            ConditionalResidualBlock(
                2 * self.ngf,
                2 * self.ngf,
                self.num_classes,
                resample      = None,
                act           = act,
                normalization = self.norm,
                dilation      = 2,
                )
            ]
        )

        self.res4 = nn.ModuleList(
            [
            ConditionalResidualBlock(
                2 * self.ngf,
                2 * self.ngf,
                self.num_classes,
                resample       = 'down',
                act            = act,
                normalization  = self.norm,
                adjust_padding = True,
                dilation       = 4,
                ),
            ConditionalResidualBlock(
                2 * self.ngf,
                2 * self.ngf,
                self.num_classes,
                resample      = None,
                act           = act,
                normalization = self.norm,
                dilation      = 4,
                )
            ]
        )


        self.refine1 = CondRefineBlock(
            [2 * self.ngf],
            2 * self.ngf,
            self.num_classes,
            self.norm,
            act   = act,
            start = True,
        )
        self.refine2 = CondRefineBlock(
            [2 * self.ngf, 2 * self.ngf],
            2 * self.ngf,
            self.num_classes, 
            self.norm,
            act = act
        )
        self.refine3 = CondRefineBlock(
            [2 * self.ngf, 2 * self.ngf],
            self.ngf,
            self.num_classes,
            self.norm,
            act = act,
        )
        self.refine4 = CondRefineBlock(
            [self.ngf, self.ngf],
            self.ngf,
            self.num_classes,
            self.norm,
            act = act,
            end = True,
        )

        self.to(device = device)

    def _compute_cond_module(self, module, x, y):
        for m in module:
            x = m(x, y)
        return x

    def forward(self, x, y):
        output = self.begin_conv(x)

        layer1 = self._compute_cond_module(self.res1, output, y)
        layer2 = self._compute_cond_module(self.res2, layer1, y)
        layer3 = self._compute_cond_module(self.res3, layer2, y)
        layer4 = self._compute_cond_module(self.res4, layer3, y)

        ref1   = self.refine1([layer4], y, layer4.shape[2:])
        ref2   = self.refine2([layer3, ref1], y, layer3.shape[2:])
        ref3   = self.refine3([layer2, ref2], y, layer2.shape[2:])
        output = self.refine4([layer1, ref3], y, layer1.shape[2:])

        output = self.normalizer(output, y)
        output = self.act(output)
        output = self.end_conv(output)
        return output



## Score Network

In [ ]:
class Model(nn.Module):
    def __init__(self, device, n_steps, sigma_min, sigma_max):
        '''
        Score network.

        L_steps   : number of perturbation schedule steps (Langevin dynamics steps).
        sigma_min : minimum sigma in the perturbation schedule.
        sigma_min : maximum sigma in the perturbation schedule.
        '''
        super().__init__()
        self.device = device
        self.sigmas = torch.exp(
            torch.linspace(
                start = math.log(sigma_max), 
                end   = math.log(sigma_min),
                steps = n_steps,
            )
        ).to(device = device)
        self.conv_layer = CondRefineNetDilated(device, n_steps)
        self.to(device  = device)

    # Loss Function
    def loss_fn(self, x, idx=None):
        '''
         The loss function is used only in the training phase.
        It performs the forward computations of the model.

        x   : Use real data if idx==None, else use perturbed data.
        idx : 'idx' must be set to 'None' during training; then, during the
              forward computation it is initialized  'idx' with random ids
              between 0 and L-1. These ids will be used to selected the of
              the noise that will perturb the data samples. 
              for inference, 'idx' must be defined with a value (not 'None').
              It is recommended that you specify 'idx'.
        '''
        scores, target, sigma = self.forward(x, idx=idx, get_target=True)
        target = target.view(target.shape[0], -1)
        scores = scores.view(scores.shape[0], -1)

        # loss = mean [(s_theta - noise/sigma)^2]*sigma^2
        losses = torch.square(scores - target).mean(dim=-1) * sigma.squeeze() ** 2
        return losses.mean(dim=0)

    # s_theta(x, sigma)
    def forward(self, x, idx=None, get_target=False):
        '''
        x          : 'x' is real data if 'idx=None', else 'x' is the perturbed data.
        idx        : 'idx' must be set to 'None' during training; then, during the
                     forward computation it is initialized  'idx' with random ids
                     between 0 and L-1. These ids will be used to selected the of
                     the noise that will perturb the data samples. 
                     for inference, 'idx' must be defined with a value (not 'None').
                     It is recommended that you specify 'idx'.
        get_target : if 'True' (training phase), the method returns 'target' and 'sigma', 
                     besides 'output' (score prediction)        '''

        # Training phase .......................................
        if idx == None:
            idx         = torch.randint(0, len(self.sigmas), (x.size(0), 1)).to(device = self.device)
            used_sigmas = self.sigmas[idx][:, :, None, None]
            noise       = torch.randn_like(x)
            x_tilde     = x + noise * used_sigmas
            idx         = idx.squeeze()

        # Sampling phase .....................................
        else:
            idx     = torch.Tensor([idx for _ in range(x.size(0))]).to(device = self.device).long()
            x_tilde = x

        if get_target:
            target = - 1 / (used_sigmas ) * noise

        output = self.conv_layer(x_tilde, idx)

        # output      = s_theta  = scores
        # target      = -z/sigma = (x_tilde - x)/sigma^2
        # used_sigmas = sigma
        return (output, target, used_sigmas) if get_target else output

## Sampling with annealed Langevin dynamics

$$\tilde{x}_{t} = \tilde{x}_{t-1}+\frac{\alpha_{i}}{2}\nabla_{\tilde{x}_{t-1}}{\log}p_{\sigma_{i}}(\tilde{x}_{t-1})+\sqrt{\alpha_{i}}z_{t}\ \ \text{where, } i\in [1, L]\ \  \text{and}\ \  t\in[1,T]$$

In [ ]:
class AnnealedLangevinDynamic():
    '''
    Class to sample a score model using annealed Langevin dynamics.
    '''
    def __init__(self, sigma_min, sigma_max, L_steps, T_steps, score_fn, device, epsilon = 1e-1):
        '''
        sigma_min : minimum variance of perturbation schedule
        sigma_max : maximum variance of perturbation schedule
        L         : iteration step of Langevin dynamics
        T         : annealed step of annealed Langevin dynamics
        score_fn  : trained score network
        epsilon   : coefficient of step size
        '''
        # Create the L variance values, regularly spaced, between the 
        # selected maximum and minimum.
        self.process = torch.exp(
            torch.linspace(
                start = math.log(sigma_max),
                end   = math.log(sigma_min),
                steps = L_steps,
            )
        )
        # alpha_i = epsilon * (sigma_i/sigma_L)^2
        self.alpha    = epsilon * (self.process / self.process[-1] ) ** 2
        self.score_fn = score_fn
        self.T_steps  = T_steps
        self.device   = device

    # One iteration of annealed step
    def _one_annealed_step_iteration(self, x, idx):
        '''
        x   : perturbated data
        idx : step of perturbation schedule
        '''
        self.score_fn.eval()
        # Draw (sampling_number x 2) values from N(0,1)
        z, alpha = torch.randn_like(x).to(device = self.device), self.alpha[idx]
        # Compute next value for samples x given current sample values and current alpha value
        x            = x + 0.5 * alpha * self.score_fn(x, idx) + torch.sqrt(alpha) * z
        return x

    # One annealed step
    def _one_annealed_step(self, x, idx):
        '''
        x   : perturbated data
        idx : step of perturbation schedule
        '''
        # For one noise level, run T annealing steps
        for _ in range(self.T_steps):
            x = self._one_annealed_step_iteration(x, idx)
        return x

    # One Langevin Step
    def _one_diffusion_step(self, x):
        '''
        x   : sampling of prior distribution
        '''
        # Iterate over the L noise variance levels
        for idx in range(len(self.process)):
            # Execute one annealed LD step
            x = self._one_annealed_step(x, idx)
            # Make 'x' an output of the iterative process
            yield x

    @torch.no_grad()
    def sampling(self, sampling_number, only_final=False):
        '''
       Draw samples from the score model using annealed Langevin dynamics.
        only_final : If True, the method returns only the output of the final schedule step.
        '''
        # Get sampling_number pairs from the prior, which is a Uniform distribution 
        # in the interval [0,1) and convert the values to the range [-1,+1].
        sample = torch.rand([sampling_number, 1, 14, 14]).to(device = self.device)
        sampling_list = []

        final = None
        for sample in self._one_diffusion_step(sample):
            final = sample
            if not only_final:
                sampling_list.append(final)


        return final if only_final else torch.stack(sampling_list)



In [ ]:
class AverageMeter(object):
    '''
    Class for keep track of a metric (the loss) during training and calculate its average.
    '''
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt  = fmt
        self.reset()

    def reset(self):
        self.val   = 0
        self.avg   = 0
        self.sum   = 0
        self.count = 0

    def update(self, val, n=1):
        self.val    = val
        self.sum   += val * n
        self.count += n
        self.avg    = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)


class ProgressMeter(object):
    '''
    Helper class to print the batch and total batches well formated during training.
    '''
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries  = [self.prefix + self.batch_fmtstr.format(batch)]
        entries += [str(meter) for meter in self.meters]

        print('\r' + '\t'.join(entries), end = '')

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt        = '{:' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

In [ ]:
def imshow(sample, sampling_number = 64):
    '''
    Plot a grid of images stores in 'samples' with size 
    sqrt(sampling_number)xsqrt(sampling_number).
    '''
    plt.figure(figsize=(10, 10))
    clear_output()
    row_number  = int(math.sqrt(sampling_number))
    col_number  = int(math.sqrt(sampling_number))
    sample      = sample[:sampling_number].detach().cpu().numpy()
    shape       = sample.shape
    show_sample = np.zeros(
        [
        row_number * shape[2],
        col_number * shape[3] ]
    ).astype(np.float32)
    for row in range(row_number):
        for col in range(col_number):
            sample_ = sample[row + col * row_number][0]
            show_sample[
                row * shape[2] : (row+1) * shape[2],
                col * shape[3] : (col+1) * shape[3] 
            ] = (sample_ - sample_.min()) / (sample_.max() - sample_.min()) * 255

    show_sample = show_sample.astype(np.uint8)
    plt.axis(False)
    plt.imshow(show_sample, cmap = 'gray')
    plt.show()

# Train Setting

## Hyperparameters of the model and perturbation process

In [ ]:
# epsilon of step size
epsilon       = 1.5e-5

# sigma min and max of Langevin dynamics
sigma_min     = 0.005
sigma_max     = 10

# Langevin dynamics steps and annealing steps
n_steps        = 10  # L
annealed_steps = 100 # T

device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

## Model, optimizer and sampler instances

In [ ]:
model   = Model(device, n_steps, sigma_min, sigma_max)
optim   = torch.optim.Adam(model.parameters(), lr = 0.005)
dynamic = AnnealedLangevinDynamic(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)

## Create the dataset and dataloader for MNIST

In [ ]:
transform = torchvision.transforms.Compose(
    [
    torchvision.transforms.Resize((14, 14)),
    torchvision.transforms.ToTensor()
    ]
)
dataset      = torchvision.datasets.MNIST(
    root      = './MNIST',
    train     = True,
    download  = True,
    transform = transform,
)
dataloader   = torch.utils.data.DataLoader(
    dataset,
    batch_size = 256,
    drop_last  = True,
)
dataiterator = iter(dataloader)


## Training hyperparameters

In [ ]:
total_iteration   = 30000
current_iteration = 0
display_iteration = 1500
sampling_number   = 16
device            = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
only_final        = True

losses   = AverageMeter('Loss', ':.4f')
progress = ProgressMeter(total_iteration, [losses], prefix='Iteration ')

## Train the Score Model

In [ ]:

while current_iteration != total_iteration:
    model.train()
    try:
        data         = next(dataiterator)
    except:
        dataiterator = iter(dataloader)
        data         = next(dataiterator)
    data = data[0].to(device = device)
    loss = model.loss_fn(data)

    optim.zero_grad()
    loss.backward()
    optim.step()

    losses.update(loss.item())
    progress.display(current_iteration)
    current_iteration += 1

    if current_iteration % display_iteration == 0:
        dynamic = AnnealedLangevinDynamic(
            sigma_min,
            sigma_max,
            n_steps,
            annealed_steps,
            model,
            device,
            epsilon = epsilon,
        )
        sample  = dynamic.sampling(sampling_number, only_final)
        imshow(sample, sampling_number)
        losses.reset()


## Sampling from The Score Model

In [ ]:
sampling_number = 4
only_final      = True
dynamic         = AnnealedLangevinDynamic(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon = epsilon,
)
sample = dynamic.sampling(sampling_number, only_final)
imshow(sample, sampling_number)

In [ ]:
def ani_imshow(sample, sampling_number = 64):

    row_number  = int(math.sqrt(sampling_number))
    col_number  = int(math.sqrt(sampling_number))
    sample      = sample[:sampling_number].detach().cpu().numpy()
    shape       = sample.shape
    show_sample = np.zeros(
        [
            row_number * shape[2],
            col_number * shape[3]
        ]
    ).astype(np.float32)
    for row in range(row_number):
        for col in range(col_number):
            sample_ = sample[row + col * row_number][0]
            show_sample[
                row * shape[2] : (row+1) * shape[2],
                col * shape[3] : (col+1) * shape[3]
            ] = (sample_ - sample_.min()) / (sample_.max() - sample_.min()) * 255

    show_sample = show_sample.astype(np.uint8)

    return show_sample

In [ ]:
sampling_number = 4
only_final      = False
dynamic         = AnnealedLangevinDynamic(
    sigma_min,
    sigma_max,
    n_steps,
    annealed_steps,
    model,
    device,
    epsilon=epsilon,
)
sample          = dynamic.sampling(sampling_number, only_final)


In [ ]:
fig = plt.figure(figsize=(8,8))
plt.axis("off")

ims = [
    [
        plt.imshow(
            ani_imshow(
                sample[i],
                sampling_number = sampling_number,
            ),
            animated=True,
            cmap = 'gray'
        )
    ] for i in range(len(sample))]
ani = animation.ArtistAnimation(
    fig,
    ims,
    interval=300,
    repeat_delay=1000,
    blit=True
)
# ani.save('ncsn_mnist.gif')

HTML(ani.to_jshtml())